# Transformer Summarization - Fine-Tuning

Fine-tuning T5 on the CNN/DailyMail dataset using HuggingFace Seq2SeqTrainer for abstractive summarization.

## Part-1:
### Importing required libraries

In [1]:
# Install required packages
!pip install transformers datasets evaluate rouge_score bert_score torch pandas -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset
from evaluate import load
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# checking GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## Loading and inspecting data

In [3]:
df = pd.read_csv('/content/cnn_dailymail_5percent.csv') # loading the dataset
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head(5))
print(f"\nMissing values:\n{df.isnull().sum()}") # checking for any missing values

Dataset shape: (15599, 3)

Columns: ['article', 'highlights', 'id']

First few rows:
                                             article  \
0  Jose Mourinho has left Thibaut Courtois sweati...   
1  Crime: Jeffrey B. Maurer, pictured, is accused...   
2  (CNN) -- Florida, already threatened with sink...   
3  A motorist was caught drink-driving - while pa...   
4  (CNN) -- Take two parts Jamie Dornan's bare to...   

                                          highlights  \
0  Chelsea face Paris St-Germain in the Champions...   
1  Jeffrey B. Maurer, 53, from Detroit Michigan w...   
2  Giant African snails are menacing Florida's Mi...   
3  John Graham, 58, left his car illegally near P...   
4  The new trailer premiered Thursday night .\nTh...   

                                         id  
0  ed2aaada35c672869c1dfb2c5678764495cafaba  
1  179843cef35f603137da7697f320388d7bc455e9  
2  fa29815cdb0aa7448cb98cce1bb3eb253590c37d  
3  4632cc2cb555de9ef8498f77bbfc09cd7319077b  
4  73b2b64c

## Keeping only required columns

In [4]:
df = df[['article', 'highlights']] # we are selecting only 'article' and 'highlights' columns
print(df)

                                                 article  \
0      Jose Mourinho has left Thibaut Courtois sweati...   
1      Crime: Jeffrey B. Maurer, pictured, is accused...   
2      (CNN) -- Florida, already threatened with sink...   
3      A motorist was caught drink-driving - while pa...   
4      (CNN) -- Take two parts Jamie Dornan's bare to...   
...                                                  ...   
15594  By . Craig Mackenzie . PUBLISHED: . 03:54 EST,...   
15595  By . Matt Chorley, Mailonline Political Editor...   
15596  Vancouver, British Columbia (CNN) -- In Februa...   
15597  Newtown, Connecticut (CNN) -- The school buses...   
15598  Lib Dem minister Norman Baker last night resig...   

                                              highlights  
0      Chelsea face Paris St-Germain in the Champions...  
1      Jeffrey B. Maurer, 53, from Detroit Michigan w...  
2      Giant African snails are menacing Florida's Mi...  
3      John Graham, 58, left his car illega

## Loading both models and tokenizers

In [5]:
def load_model_for_summarization(model_id, device): # loading both models base and fine tuned
    print(f"Loading model: {model_id}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device).eval()# chain the .to(device) and .eval() calls
        print(f"Successfully loaded {model_id} to {device}.\n")
        return tokenizer, model
    except Exception as e:
        print(f"Failed to load {model_id}. Error: {e}\n")
        return None, None

base_model_id = "google-t5/t5-small" # defining model names
finetuned_model_id = "ubikpt/t5-small-finetuned-cnn"
# loading base model
base_tokenizer, base_model = load_model_for_summarization(base_model_id, device)
# loading fine-tuned model
finetuned_tokenizer, finetuned_model = load_model_for_summarization(finetuned_model_id, device)

if base_model and finetuned_model:
    print("All models are ready.")
else:
    print("Error: One or more models failed to load.")

Loading model: google-t5/t5-small...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

Successfully loaded ubikpt/t5-small-finetuned-cnn to cuda.

All models are ready.


## Preprocessing and tokenizing

In [6]:
prefix = "summarize: " # defining ONE preprocessing function

def preprocess_function(examples, tokenizer):
    inputs = [prefix + doc for doc in examples["article"]] # adding the summarization prefix and tokenizing
    model_inputs = tokenizer(inputs, max_length=1024, padding=True, truncation=True)
    labels = tokenizer(examples["highlights"], max_length=128, padding=True, truncation=True)   # tokenizing the summaries (labels)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


dataset = Dataset.from_pandas(df) # converting pandas to HuggingFace dataset
print(f"Dataset created with {len(dataset)} examples")

print("\nTokenizing for base model")  # tokenizing for base model
tokenized_ds1 = dataset.map(lambda x: preprocess_function(x, base_tokenizer),
    batched=True,
    remove_columns=dataset.column_names)
tokenized_ds1.set_format("torch")
print(f"Tokenized dataset 1: {tokenized_ds1}")

print("\nTokenizing for fine tuned model") # tokenizing for fine tuned model
tokenized_ds2 = dataset.map(lambda x: preprocess_function(x, finetuned_tokenizer),
    batched=True,
    remove_columns=dataset.column_names)
tokenized_ds2.set_format("torch")
print(f"Tokenized dataset 2: {tokenized_ds2}")

Dataset created with 15599 examples

Tokenizing for base model


Map:   0%|          | 0/15599 [00:00<?, ? examples/s]

Tokenized dataset 1: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 15599
})

Tokenizing for fine tuned model


Map:   0%|          | 0/15599 [00:00<?, ? examples/s]

Tokenized dataset 2: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 15599
})


## Generating Summaries

In [7]:
def generate_batch_summaries_WITH_PREFIX(batch): # function for base model (with prefix)
    inputs = [prefix + article for article in batch['article']]
    tokenized = base_tokenizer(inputs, max_length=1024, padding="longest", truncation=True, return_tensors="pt")
    tokenized = {k: v.to(device) for k, v in tokenized.items()}
    with torch.no_grad():
        outputs = base_model.generate(**tokenized, max_length=128, num_beams=4, early_stopping=True)
    return {"gen_summary": base_tokenizer.batch_decode(outputs, skip_special_tokens=True)}

def generate_batch_summaries_NO_PREFIX(batch): # function for fine tuned model (no prefix)
    inputs = batch['article'] # No prefix
    tokenized = finetuned_tokenizer(inputs, max_length=1024, padding="longest", truncation=True, return_tensors="pt")
    tokenized = {k: v.to(device) for k, v in tokenized.items()}
    with torch.no_grad():
        outputs = finetuned_model.generate(**tokenized, max_length=128, num_beams=4, early_stopping=True)
    return {"gen_summary": finetuned_tokenizer.batch_decode(outputs, skip_special_tokens=True)}


references = dataset['highlights']
print(f"Generating summaries for {len(dataset)} articles...\n")

print("Model 1 (google-t5/t5-small) generation (with prefix):") # generating for base model
dataset_with_summaries_1 = dataset.map(generate_batch_summaries_WITH_PREFIX,batched=True, batch_size=8)
predictions_model1 = dataset_with_summaries_1['gen_summary']
print(f"Generated {len(predictions_model1)} summaries\n")


print("Model 2 (ubikpt/t5-small-finetuned-cnn) generation (no prefix):") # generating for fine tuned model
dataset_with_summaries_2 = dataset.map(generate_batch_summaries_NO_PREFIX,batched=True,batch_size=8)
predictions_model2 = dataset_with_summaries_2['gen_summary']
print(f"Generated {len(predictions_model2)} summaries")

Generating summaries for 15599 articles...

Model 1 (google-t5/t5-small) generation (with prefix):


Map:   0%|          | 0/15599 [00:00<?, ? examples/s]

Generated 15599 summaries

Model 2 (ubikpt/t5-small-finetuned-cnn) generation (no prefix):


Map:   0%|          | 0/15599 [00:00<?, ? examples/s]

Generated 15599 summaries


## Evaluating ROUGE, Perplexity, BERTSCORE

In [10]:
references = dataset["highlights"]

rouge = load("rouge") # ROUGE
rouge_results_1= rouge.compute(predictions=predictions_model1, references=references)
rouge_results_2 = rouge.compute(predictions=predictions_model2, references=references)
print("rouge_results_model1:",rouge_results_1)
print("\nrouge_results_model2:",rouge_results_2)

perplexity = load("perplexity", module_type="metric") # Perplexity
perp_results_1 = perplexity.compute(model_id="gpt2", predictions=predictions_model1)
perp_results_2 = perplexity.compute(model_id="gpt2", predictions=predictions_model2)
print("Perplexity_results_model1:",perp_results_1['perplexities'][:10]) # only 10 values
print("Perplexity_results_model1:",perp_results_2['perplexities'][:10])

bertscore = load("bertscore") # BERTSCORE
bert_results_1 = bertscore.compute(predictions=predictions_model1, references=references, lang="en")
bert_results_2 = bertscore.compute(predictions=predictions_model2, references=references, lang="en")

def summarize_bert_scores(results):
    return {"Precision": np.mean(results["precision"]),
        "Recall": np.mean(results["recall"]),
        "F1": np.mean(results["f1"]),}

bert_summary_1 = summarize_bert_scores(bert_results_1)
bert_summary_2 = summarize_bert_scores(bert_results_2)

print("\nBERTSCORE Summary-")
print("Model 1:", bert_summary_1)
print("Model 2:", bert_summary_2)

model_names = ["google-t5/t5-small", "ubikpt-finetuned-cnn"] # extract final metrics

rouge1_scores = [rouge_results_1["rouge1"], rouge_results_2["rouge1"]]
rouge2_scores = [rouge_results_1["rouge2"], rouge_results_2["rouge2"]]
rougeL_scores = [rouge_results_1["rougeLsum"], rouge_results_2["rougeLsum"]]
perplexities = [perp_results_1["mean_perplexity"], perp_results_2["mean_perplexity"]]
bert_f1_scores = [bert_summary_1["F1"], bert_summary_2["F1"]]

print("\n Final evaluation summary")
print(f"{'Model':<25} {'ROUGE-1':>8} {'ROUGE-2':>8} {'ROUGE-L':>8} {'Perplexity':>12} {'BERT F1':>9}")
print("-" * 75)
for i in range(2):
    print(f"{model_names[i]:<25} "
          f"{rouge1_scores[i]:>8.4f} "
          f"{rouge2_scores[i]:>8.4f} "
          f"{rougeL_scores[i]:>8.4f} "
          f"{perplexities[i]:>12.2f} "
          f"{bert_f1_scores[i]:>9.4f}")



rouge_results_model1: {'rouge1': np.float64(0.3427148239437285), 'rouge2': np.float64(0.1497359998044846), 'rougeL': np.float64(0.2492533030383109), 'rougeLsum': np.float64(0.3000359659724131)}

rouge_results_model2: {'rouge1': np.float64(0.2905909035061399), 'rouge2': np.float64(0.13426263674419348), 'rougeL': np.float64(0.22796473950439333), 'rougeLsum': np.float64(0.26387014074244247)}


  0%|          | 0/975 [00:00<?, ?it/s]

  0%|          | 0/975 [00:00<?, ?it/s]

Perplexity_results_model1: [16.822341918945312, 17.41632080078125, 26.86276626586914, 54.218746185302734, 30.43648910522461, 132.26438903808594, 47.91737747192383, 138.49185180664062, 55.78977966308594, 47.944313049316406]
Perplexity_results_model1: [36.02077102661133, 20.468597412109375, 43.59503936767578, 80.97413635253906, 46.07203674316406, 139.9925079345703, 108.44571685791016, 50.23685836791992, 88.99470520019531, 132.73204040527344]


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



BERTSCORE Summary-
Model 1: {'Precision': np.float64(0.8737117948727131), 'Recall': np.float64(0.8553282154340822), 'F1': np.float64(0.8642941071434016)}
Model 2: {'Precision': np.float64(0.8870902665494856), 'Recall': np.float64(0.8518118597550365), 'F1': np.float64(0.868899415949918)}

 Final evaluation summary
Model                      ROUGE-1  ROUGE-2  ROUGE-L   Perplexity   BERT F1
---------------------------------------------------------------------------
google-t5/t5-small          0.3427   0.1497   0.3000        54.49    0.8643
ubikpt-finetuned-cnn        0.2906   0.1343   0.2639        98.00    0.8689


## Manual comparison of 2 Summaries

In [12]:
sample_indices = [3, 5]
for idx in sample_indices:
    print(f"\n\nArticle {idx+1}-")
    print("\nOriginal article (first 500 characters):")
    print(df['article'][idx][:500] + "...")
    print("\nGround truth summary:")
    print(df['highlights'][idx])
    print("\nModel 1 (google-t5/t5-small) summary:")
    print(predictions_model1[idx])
    print("\nModel 2 (ubikpt/t5-small-finetuned-cnn) summary:")
    print(predictions_model2[idx])



Article 4-

Original article (first 500 characters):
A motorist was caught drink-driving - while parking his car near a court where he was due to stand trial. John Graham, 58, left his Toyota MR2 illegally near Plymouth Magistrates' Court, where he was supposed to face a jury on charges of fraud. But a passing police officer spotted the vehicle obstructing an entrance and Graham failed a breath test. John Graham, 58, was caught drink-driving parking his car near a court where he was due to stand trial . He then appeared in two courts on the same d...

Ground truth summary:
John Graham, 58, left his car illegally near Plymouth Magistrates' Court .
He was supposed to face a jury on charges on fraud .
But a police officer spotted the vehicle and Graham failed a breath test .

Model 1 (google-t5/t5-small) summary:
the 58-year-old left his Toyota MR2 illegally near a court. he was supposed to face a jury on charges of fraud. but passing police officer spotted the vehicle obstructing an en